# Silver Table 13: project-to-region mapping

## Purpose

Publish exactly one deterministic region-mapping result for every canonical `silver_project`. The notebook preserves non-geographic, ambiguous, unmatched, conflicting, and invalid outcomes instead of dropping or force-matching them. It reads the implemented Silver project and PSGC contracts, approved configuration, compact audit evidence, and the selected Bronze region boundaries.


## Step 1 — Prove the upstream state is safe

The gate requires one nonempty project run, one current PSGC run, a successful and validated boundary snapshot, one active approved version per optional mapping domain, and valid approved PSGC targets. Empty alias and manual-exception tables are allowed.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

CREATE OR REPLACE TEMPORARY VIEW latest_boundary_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY table_name
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'boundaries'
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW latest_boundary_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.dq_results
    WHERE table_name = 'boundaries'
      AND snapshot_id = (SELECT snapshot_id FROM latest_boundary_load)
)
WHERE validation_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_project_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND pipeline_run_id = (SELECT MAX(run_id) FROM `02-silver`.silver_project);

CREATE OR REPLACE TEMPORARY VIEW selected_psgc_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name = 'silver_psgc_place'
  AND pipeline_run_id = (
      SELECT MAX(run_id) FROM `02-silver`.silver_psgc_place WHERE place_type = 'REGION'
  );

CREATE OR REPLACE TEMPORARY VIEW latest_config_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 1
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 1
    AND COUNT_IF(rows_loaded > 0) = 1,
    'STOP: latest boundary Bronze load is not safe'
)
FROM latest_boundary_load;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest boundary Bronze validation is unsafe'
)
FROM latest_boundary_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM selected_project_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: selected Silver PSGC validation is missing or unsafe'
)
FROM selected_psgc_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: latest Silver configuration validation is missing or unsafe'
)
FROM latest_config_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key),
    'STOP: silver_project must be nonempty and unique in one selected run'
)
FROM `02-silver`.silver_project;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(DISTINCT psgc_version) <= 1
    AND COUNT(*) = COUNT(DISTINCT psgc_code),
    'STOP: selected official PSGC region set is empty or unsafe'
)
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'REGION';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT _source_snapshot_id) = 1
    AND COUNT(DISTINCT _ingest_run_id) = 1
    AND MAX(_source_snapshot_id) = (SELECT snapshot_id FROM latest_boundary_load),
    'STOP: current region boundaries do not match the validated boundary snapshot'
)
FROM `01-bronze`.boundaries
WHERE LOWER(TRIM(administrative_level)) = 'region';

SELECT ASSERT_TRUE(
    COUNT(DISTINCT alias_version) <= 1,
    'STOP: multiple active approved DPWH region-alias versions are unsafe'
)
FROM `02-silver`.config_place_name_alias
WHERE source_system = 'DPWH'
  AND UPPER(TRIM(place_type)) = 'REGION'
  AND approval_status = 'APPROVED' AND is_active = TRUE;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT mapping_version) <= 1,
    'STOP: multiple active approved DPWH project-override versions are unsafe'
)
FROM `02-silver`.config_manual_geographic_match
WHERE source_system = 'DPWH'
  AND UPPER(TRIM(record_type)) = 'PROJECT'
  AND approval_status = 'APPROVED' AND is_active = TRUE;


## Step 2 — Select deterministic versions and prepare source contracts

Name normalization is deliberately conservative: `TRIM`, uppercase, and repeated-whitespace collapse only. The current project contract has no legitimate project PSGC code, so that stage remains explicitly not applicable. Source load time comes from Bronze audit lineage; publisher version remains nullable.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_region_mapping_parameters AS
SELECT
    CAST(4.2 AS DOUBLE) AS min_latitude,
    CAST(21.3 AS DOUBLE) AS max_latitude,
    CAST(116.0 AS DOUBLE) AS min_longitude,
    CAST(127.0 AS DOUBLE) AS max_longitude;

CREATE OR REPLACE TEMPORARY VIEW selected_project_region_versions AS
WITH project_context AS (
    SELECT
        MAX(run_id) AS project_run_id,
        MAX(source_snapshot_id) AS project_source_snapshot_id,
        MAX(source_ingest_run_id) AS project_source_ingest_run_id,
        MAX(transformation_rule_version) AS project_rule_version
    FROM `02-silver`.silver_project
),
psgc_context AS (
    SELECT
        MAX(run_id) AS psgc_run_id,
        MAX(source_snapshot_id) AS psgc_source_snapshot_id,
        MAX(source_ingest_run_id) AS psgc_source_ingest_run_id,
        MAX(psgc_version) AS psgc_version,
        MAX(transformation_rule_version) AS psgc_rule_version,
        MAX(source_load_ts) AS psgc_source_load_ts
    FROM `02-silver`.silver_psgc_place
    WHERE place_type = 'REGION'
),
boundary_context AS (
    SELECT
        MAX(_source_snapshot_id) AS boundary_source_snapshot_id,
        MAX(_ingest_run_id) AS boundary_source_ingest_run_id,
        MAX(source_psgc_version) AS boundary_psgc_version,
        MAX(source_boundary_version) AS boundary_version,
        MAX(_source_system) AS boundary_source_system,
        MAX(_ingested_at) AS boundary_source_load_ts
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN latest_boundary_load AS latest
    WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
      AND boundary._source_snapshot_id = latest.snapshot_id
),
alias_context AS (
    SELECT COALESCE(MAX(alias_version), 'NO_APPROVED_ALIAS_VERSION') AS alias_version
    FROM `02-silver`.config_place_name_alias
    WHERE source_system = 'DPWH'
      AND UPPER(TRIM(place_type)) = 'REGION'
      AND approval_status = 'APPROVED' AND is_active = TRUE
),
manual_context AS (
    SELECT COALESCE(MAX(mapping_version), 'NO_APPROVED_MANUAL_VERSION') AS manual_mapping_version
    FROM `02-silver`.config_manual_geographic_match
    WHERE source_system = 'DPWH'
      AND UPPER(TRIM(record_type)) = 'PROJECT'
      AND approval_status = 'APPROVED' AND is_active = TRUE
)
SELECT
    project_context.*, psgc_context.*, boundary_context.*,
    alias_context.alias_version, manual_context.manual_mapping_version,
    'silver-project-region-map-v1' AS mapping_rule_version,
    SHA2(CONCAT_WS(
        '||',
        project_context.project_run_id, project_context.project_source_snapshot_id,
        psgc_context.psgc_run_id, psgc_context.psgc_source_snapshot_id,
        COALESCE(psgc_context.psgc_version, 'NO_PUBLISHER_VERSION'),
        boundary_context.boundary_source_snapshot_id,
        COALESCE(boundary_context.boundary_version, 'NO_BOUNDARY_VERSION'),
        boundary_context.boundary_source_ingest_run_id,
        alias_context.alias_version, manual_context.manual_mapping_version,
        'silver-project-region-map-v1'
    ), 256) AS mapping_run_id
FROM project_context
CROSS JOIN psgc_context
CROSS JOIN boundary_context
CROSS JOIN alias_context
CROSS JOIN manual_context;

CREATE OR REPLACE TEMPORARY VIEW project_load_lineage AS
SELECT run_id AS source_ingest_run_id, source_version,
       COALESCE(completed_at, started_at) AS source_load_ts
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS audit_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'dpwh_projects'
      AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
)
WHERE audit_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW project_region_input AS
SELECT
    'DPWH' AS source_system, project.project_key, project.contract_id,
    project.reported_region AS reported_region_raw,
    REGEXP_REPLACE(UPPER(TRIM(project.reported_region)), '\\s+', ' ') AS reported_region_standardized,
    project.latitude_parsed AS latitude, project.longitude_parsed AS longitude,
    CASE
        WHEN project.latitude_parsed IS NULL AND project.longitude_parsed IS NULL THEN 'MISSING_PAIR'
        WHEN project.latitude_parsed IS NULL OR project.longitude_parsed IS NULL THEN 'PARTIAL_PAIR'
        WHEN project.latitude_parsed NOT BETWEEN parameters.min_latitude AND parameters.max_latitude
          OR project.longitude_parsed NOT BETWEEN parameters.min_longitude AND parameters.max_longitude THEN 'OUT_OF_RANGE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,
    project.source_snapshot_id AS project_source_snapshot_id,
    project.source_ingest_run_id AS project_source_ingest_run_id,
    project.run_id AS project_run_id,
    project.transformation_rule_version AS project_rule_version,
    project.source_modified_at AS project_source_modified_at,
    load.source_version AS project_source_version,
    load.source_load_ts
FROM `02-silver`.silver_project AS project
CROSS JOIN project_region_mapping_parameters AS parameters
LEFT JOIN project_load_lineage AS load
  ON project.source_ingest_run_id = load.source_ingest_run_id;

CREATE OR REPLACE TEMPORARY VIEW official_project_regions AS
SELECT
    psgc_code, place_name_raw, place_name_standardized, psgc_version,
    source_snapshot_id, source_ingest_run_id, run_id, transformation_rule_version, source_load_ts
FROM `02-silver`.silver_psgc_place
WHERE place_type = 'REGION';


## Step 3 — Build reviewed, exact-name, alias, and spatial candidates

The automatic waterfall is deterministic. `Central Office` is removed before name, alias, and spatial stages; approved manual evidence is retained only to expose an ignored override conflict. Manual exceptions and aliases must be active and `APPROVED`. Exact names preserve meaningful region tokens and punctuation. Spatial evidence uses only the selected region polygons, native Databricks SQL geometry, longitude-first points, and no fuzzy, nearest-neighbor, or Python logic. The source GeoJSON stores numeric coordinates as quoted strings, so SQL removes quotes only around numeric coordinate tokens before safe parsing.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW approved_manual_project_targets AS
SELECT manual.source_record_id AS contract_id, manual.canonical_psgc_code,
       manual.canonical_place_name, manual.override_reason, manual.mapping_version,
       manual.source_reference, manual.approved_by, manual.approved_at,
       target.psgc_code AS verified_target_code, target.place_type AS verified_target_type,
       CASE WHEN target.place_type = 'REGION' THEN target.psgc_code ELSE target.region_code END
           AS canonical_region_code
FROM `02-silver`.config_manual_geographic_match AS manual
CROSS JOIN selected_project_region_versions AS selected
LEFT JOIN `02-silver`.silver_psgc_place AS target
  ON manual.canonical_psgc_code = target.psgc_code
 AND target.source_snapshot_id = selected.psgc_source_snapshot_id
WHERE manual.source_system = 'DPWH'
  AND UPPER(TRIM(manual.record_type)) = 'PROJECT'
  AND manual.approval_status = 'APPROVED' AND manual.is_active = TRUE
  AND manual.mapping_version = selected.manual_mapping_version;

CREATE OR REPLACE TEMPORARY VIEW approved_project_region_aliases AS
SELECT
    REGEXP_REPLACE(UPPER(TRIM(alias.raw_place_name)), '\\s+', ' ') AS raw_region_standardized,
    alias.canonical_psgc_code, alias.canonical_place_name, alias.match_rule, alias.alias_version
FROM `02-silver`.config_place_name_alias AS alias
CROSS JOIN selected_project_region_versions AS selected
WHERE alias.source_system = 'DPWH'
  AND UPPER(TRIM(alias.place_type)) = 'REGION'
  AND COALESCE(TRIM(alias.raw_region_name), '') = ''
  AND COALESCE(TRIM(alias.raw_province_name), '') = ''
  AND COALESCE(TRIM(alias.raw_city_municipality_name), '') = ''
  AND alias.approval_status = 'APPROVED' AND alias.is_active = TRUE
  AND alias.alias_version = selected.alias_version;

SELECT ASSERT_TRUE(
    COUNT_IF(verified_target_code IS NULL OR canonical_region_code IS NULL) = 0,
    'STOP: an approved manual project mapping cannot resolve to one selected official region'
)
FROM approved_manual_project_targets;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'STOP: duplicate active approved overrides exist for one project')
FROM (
    SELECT contract_id FROM approved_manual_project_targets
    GROUP BY contract_id HAVING COUNT(*) > 1
);

SELECT ASSERT_TRUE(COUNT(*) = 0, 'STOP: an approved override references no canonical project')
FROM approved_manual_project_targets AS manual
LEFT ANTI JOIN project_region_input AS project ON manual.contract_id = project.contract_id;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'STOP: an approved project region alias targets no selected official region')
FROM approved_project_region_aliases AS alias
LEFT ANTI JOIN official_project_regions AS region
  ON alias.canonical_psgc_code = region.psgc_code;

SELECT ASSERT_TRUE(COUNT(*) = 0, 'STOP: one approved region alias resolves to multiple regions')
FROM (
    SELECT raw_region_standardized
    FROM approved_project_region_aliases
    GROUP BY raw_region_standardized
    HAVING COUNT(DISTINCT canonical_psgc_code) > 1
);

CREATE OR REPLACE TEMPORARY VIEW manual_region_candidates AS
SELECT project.project_key,
       COUNT(DISTINCT manual.canonical_region_code) AS manual_candidate_count,
       COLLECT_SET(manual.canonical_region_code) AS manual_candidate_codes,
       CASE WHEN COUNT(DISTINCT manual.canonical_region_code) = 1
            THEN ELEMENT_AT(SORT_ARRAY(COLLECT_SET(manual.canonical_region_code)), 1)
       END AS manual_region_code,
       CONCAT_WS('; ', SORT_ARRAY(COLLECT_SET(manual.override_reason))) AS manual_override_reason,
       MAX(manual.canonical_psgc_code) AS manual_target_psgc_code,
       MAX(manual.source_reference) AS manual_source_reference,
       MAX(manual.approved_by) AS manual_approved_by,
       MAX(manual.approved_at) AS manual_approved_at
FROM project_region_input AS project
JOIN approved_manual_project_targets AS manual ON project.contract_id = manual.contract_id
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW exact_region_candidates AS
SELECT project.project_key,
       COUNT(DISTINCT region.psgc_code) AS exact_candidate_count,
       COLLECT_SET(region.psgc_code) AS exact_candidate_codes,
       CASE WHEN COUNT(DISTINCT region.psgc_code) = 1
            THEN ELEMENT_AT(SORT_ARRAY(COLLECT_SET(region.psgc_code)), 1) END AS exact_region_code
FROM project_region_input AS project
JOIN official_project_regions AS region
  ON project.reported_region_standardized = region.place_name_standardized
WHERE project.reported_region_standardized <> 'CENTRAL OFFICE'
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW alias_region_candidates AS
SELECT project.project_key,
       COUNT(DISTINCT region.psgc_code) AS alias_candidate_count,
       COLLECT_SET(region.psgc_code) AS alias_candidate_codes,
       CASE WHEN COUNT(DISTINCT region.psgc_code) = 1
            THEN ELEMENT_AT(SORT_ARRAY(COLLECT_SET(region.psgc_code)), 1) END AS alias_region_code
FROM project_region_input AS project
JOIN approved_project_region_aliases AS alias
  ON project.reported_region_standardized = alias.raw_region_standardized
JOIN official_project_regions AS region ON alias.canonical_psgc_code = region.psgc_code
WHERE project.reported_region_standardized <> 'CENTRAL OFFICE'
GROUP BY project.project_key;

CREATE OR REPLACE TEMPORARY VIEW selected_region_boundary_profile AS
SELECT
    boundary.source_feature_id, boundary.psgc_code AS psgc_region_code,
    boundary.source_psgc_version AS boundary_psgc_version,
    boundary.source_boundary_version AS boundary_version,
    boundary._source_snapshot_id AS boundary_source_snapshot_id,
    boundary._ingest_run_id AS boundary_source_ingest_run_id,
    boundary._source_system AS boundary_source_system,
    boundary._ingested_at AS boundary_source_load_ts,
    region.psgc_code AS resolved_current_region_code,
    TRY_TO_GEOMETRY(REGEXP_REPLACE(
        REGEXP_REPLACE(
            boundary.geometry_json,
            '\"(\\[[^\"]*\\])\"',
            CONCAT(CHAR(36), '1')
        ),
        '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
        CONCAT(CHAR(36), '1')
    )) AS boundary_geometry
FROM `01-bronze`.boundaries AS boundary
LEFT JOIN official_project_regions AS region ON boundary.psgc_code = region.psgc_code
CROSS JOIN selected_project_region_versions AS selected
WHERE LOWER(TRIM(boundary.administrative_level)) = 'region'
  AND boundary._source_snapshot_id = selected.boundary_source_snapshot_id;

CREATE OR REPLACE TEMPORARY VIEW selected_region_boundaries AS
SELECT *
FROM selected_region_boundary_profile
WHERE resolved_current_region_code IS NOT NULL;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(boundary_geometry IS NULL) = 0
    AND COUNT(*) = COUNT(DISTINCT psgc_region_code),
    'STOP: selected region boundary set is empty, invalid, or duplicated'
)
FROM selected_region_boundaries;

CREATE OR REPLACE TEMPORARY VIEW spatial_region_candidates AS
SELECT /*+ BROADCAST(boundary) */
    project.project_key,
    COUNT(DISTINCT boundary.psgc_region_code) AS spatial_candidate_count,
    COUNT(DISTINCT boundary.source_feature_id) AS boundary_feature_count,
    COLLECT_SET(boundary.psgc_region_code) AS spatial_candidate_codes,
    CASE
        WHEN COUNT(DISTINCT boundary.psgc_region_code) = 1
            THEN ELEMENT_AT(
                SORT_ARRAY(COLLECT_SET(boundary.psgc_region_code)),
                1
            )
    END AS spatial_region_code
FROM project_region_input AS project
JOIN selected_region_boundaries AS boundary
    ON ST_COVERS(
        boundary.boundary_geometry,
        ST_POINT(project.longitude, project.latitude, 4326)
    )
WHERE project.coordinate_status = 'VALID_PAIR'
  AND project.reported_region_standardized <> 'CENTRAL OFFICE'
GROUP BY project.project_key;


## Step 4 — Apply precedence and publish one final row per project

Precedence is: non-geographic Central Office, approved manual exception, legitimate project PSGC code when available (not available in the current contract), exact standardized region name, approved alias, coordinate-boundary fallback, then unresolved. A higher-priority accepted region remains authoritative when a coordinate cross-check disagrees, but the final status becomes `MATCHED_WITH_CONFLICT`.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_region_mapping_decisions AS
WITH candidates AS (
    SELECT
        project.*,
        selected.psgc_run_id, selected.psgc_source_snapshot_id,
        selected.psgc_source_ingest_run_id, selected.psgc_version,
        selected.psgc_rule_version, selected.psgc_source_load_ts,
        selected.boundary_source_snapshot_id, selected.boundary_source_ingest_run_id,
        selected.boundary_psgc_version, selected.boundary_version,
        selected.boundary_source_system, selected.boundary_source_load_ts,
        selected.alias_version, selected.manual_mapping_version,
        selected.mapping_rule_version, selected.mapping_run_id,
        COALESCE(manual.manual_candidate_count, 0) AS manual_candidate_count,
        COALESCE(manual.manual_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS manual_candidate_codes,
        manual.manual_region_code, manual.manual_override_reason,
        manual.manual_target_psgc_code, manual.manual_source_reference,
        manual.manual_approved_by, manual.manual_approved_at,
        COALESCE(exact.exact_candidate_count, 0) AS exact_candidate_count,
        COALESCE(exact.exact_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS exact_candidate_codes,
        exact.exact_region_code,
        COALESCE(alias.alias_candidate_count, 0) AS alias_candidate_count,
        COALESCE(alias.alias_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS alias_candidate_codes,
        alias.alias_region_code,
        COALESCE(spatial.spatial_candidate_count, 0) AS spatial_candidate_count,
        COALESCE(spatial.boundary_feature_count, 0) AS boundary_feature_count,
        COALESCE(spatial.spatial_candidate_codes, CAST(ARRAY() AS ARRAY<STRING>))
            AS spatial_candidate_codes,
        spatial.spatial_region_code
    FROM project_region_input AS project
    CROSS JOIN selected_project_region_versions AS selected
    LEFT JOIN manual_region_candidates AS manual USING (project_key)
    LEFT JOIN exact_region_candidates AS exact USING (project_key)
    LEFT JOIN alias_region_candidates AS alias USING (project_key)
    LEFT JOIN spatial_region_candidates AS spatial USING (project_key)
),
selected_stage AS (
    SELECT *,
        CASE
            WHEN reported_region_standardized = 'CENTRAL OFFICE' THEN 'NON_GEOGRAPHIC_RULE'
            WHEN manual_candidate_count > 0 THEN 'MANUAL_OVERRIDE'
            WHEN exact_candidate_count > 0 THEN 'EXACT_REGION_NAME'
            WHEN alias_candidate_count > 0 THEN 'APPROVED_REGION_ALIAS'
            WHEN spatial_candidate_count > 0 THEN 'COORDINATE_BOUNDARY'
            ELSE 'NONE'
        END AS mapping_method,
        CASE
            WHEN reported_region_standardized = 'CENTRAL OFFICE' THEN NULL
            WHEN manual_candidate_count = 1 THEN manual_region_code
            WHEN manual_candidate_count > 1 THEN NULL
            WHEN exact_candidate_count = 1 THEN exact_region_code
            WHEN exact_candidate_count > 1 THEN NULL
            WHEN alias_candidate_count = 1 THEN alias_region_code
            WHEN alias_candidate_count > 1 THEN NULL
            WHEN spatial_candidate_count = 1 THEN spatial_region_code
            ELSE NULL
        END AS accepted_region_code,
        CASE
            WHEN reported_region_standardized = 'CENTRAL OFFICE' THEN 0
            WHEN manual_candidate_count > 0 THEN manual_candidate_count
            WHEN exact_candidate_count > 0 THEN exact_candidate_count
            WHEN alias_candidate_count > 0 THEN alias_candidate_count
            ELSE spatial_candidate_count
        END AS final_candidate_count
    FROM candidates
),
classified AS (
    SELECT *,
        CASE
            WHEN project_key IS NULL OR contract_id IS NULL OR TRIM(contract_id) = '' THEN 'INVALID_SOURCE'
            WHEN mapping_method = 'NON_GEOGRAPHIC_RULE' THEN 'NON_GEOGRAPHIC'
            WHEN final_candidate_count > 1 THEN 'AMBIGUOUS'
            WHEN accepted_region_code IS NULL THEN 'UNMATCHED'
            WHEN mapping_method IN ('MANUAL_OVERRIDE', 'EXACT_REGION_NAME', 'APPROVED_REGION_ALIAS')
             AND spatial_candidate_count = 1
             AND accepted_region_code <> spatial_region_code THEN 'MATCHED_WITH_CONFLICT'
            ELSE 'MATCHED'
        END AS match_status,
        CASE
            WHEN mapping_method IN ('NON_GEOGRAPHIC_RULE', 'MANUAL_OVERRIDE') THEN 'REVIEWED'
            WHEN mapping_method = 'EXACT_REGION_NAME' THEN 'EXACT'
            WHEN mapping_method = 'APPROVED_REGION_ALIAS' THEN 'APPROVED_ALIAS'
            WHEN mapping_method = 'COORDINATE_BOUNDARY' AND final_candidate_count = 1 THEN 'SPATIAL'
            WHEN final_candidate_count > 1 THEN 'REVIEW_REQUIRED'
            ELSE 'UNRESOLVED'
        END AS match_quality,
        CASE
            WHEN mapping_method = 'NON_GEOGRAPHIC_RULE' THEN 'NOT_APPLICABLE_NON_GEOGRAPHIC'
            WHEN coordinate_status <> 'VALID_PAIR' THEN 'NOT_AVAILABLE'
            WHEN spatial_candidate_count = 0 THEN 'NO_BOUNDARY_MATCH'
            WHEN spatial_candidate_count > 1 THEN 'AMBIGUOUS_BOUNDARY'
            WHEN accepted_region_code = spatial_region_code THEN 'REGION_MATCH'
            WHEN accepted_region_code IS NOT NULL THEN 'REGION_MISMATCH'
            ELSE 'SPATIAL_ONLY'
        END AS coordinate_crosscheck_status
    FROM selected_stage
)
SELECT *,
    CASE mapping_method
        WHEN 'NON_GEOGRAPHIC_RULE' THEN 'CENTRAL_OFFICE_EXACT_RULE'
        WHEN 'MANUAL_OVERRIDE' THEN COALESCE(manual_override_reason, 'APPROVED_MANUAL_EXCEPTION')
        WHEN 'EXACT_REGION_NAME' THEN 'CONSERVATIVE_STANDARDIZED_NAME_MATCH'
        WHEN 'APPROVED_REGION_ALIAS' THEN 'APPROVED_ALIAS_MATCH'
        WHEN 'COORDINATE_BOUNDARY' THEN 'REGION_BOUNDARY_POINT_COVERAGE'
        ELSE CASE
            WHEN reported_region_raw IS NULL OR TRIM(reported_region_raw) = '' THEN 'MISSING_REPORTED_REGION'
            WHEN coordinate_status <> 'VALID_PAIR' THEN 'NO_ACCEPTED_NAME_AND_NO_USABLE_COORDINATE'
            ELSE 'NO_ACCEPTED_REGION_CANDIDATE'
        END
    END AS mapping_reason
FROM classified;

CREATE OR REPLACE TABLE `02-silver`.silver_project_region_map
USING DELTA
COMMENT 'One deterministic region-mapping result per canonical project and mapping run'
AS
SELECT
    SHA2(CONCAT_WS('|', decision.source_system, decision.project_key, decision.mapping_run_id), 256)
        AS project_region_map_key,
    decision.source_system, decision.project_key, decision.contract_id,
    decision.reported_region_raw, decision.reported_region_standardized,
    decision.latitude, decision.longitude, decision.coordinate_status,
    'PROJECT_PSGC_CODE_NOT_AVAILABLE' AS project_psgc_code_stage_status,
    region.psgc_code AS psgc_region_code,
    region.place_name_raw AS psgc_region_name,
    decision.mapping_method, decision.match_status, decision.match_quality,
    decision.final_candidate_count AS candidate_count,
    CASE WHEN decision.match_status = 'AMBIGUOUS'
         THEN CONCAT(decision.mapping_method, '_MULTIPLE_REGION_CANDIDATES') END
        AS ambiguity_reason,
    CASE
        WHEN decision.match_status = 'INVALID_SOURCE' THEN 'MISSING_STABLE_SOURCE_IDENTITY'
        WHEN decision.mapping_method = 'NON_GEOGRAPHIC_RULE'
         AND decision.manual_candidate_count > 0 THEN 'CENTRAL_OFFICE_OVERRIDE_IGNORED'
        WHEN decision.match_status = 'MATCHED_WITH_CONFLICT' THEN 'NAME_COORDINATE_REGION_MISMATCH'
        WHEN decision.match_status IN ('AMBIGUOUS', 'UNMATCHED') THEN decision.mapping_reason
    END AS exception_reason,
    decision.mapping_reason, decision.coordinate_crosscheck_status,
    decision.spatial_region_code AS coordinate_region_code,
    spatial_region.place_name_raw AS coordinate_region_name,
    decision.final_candidate_count, decision.manual_candidate_count,
    decision.exact_candidate_count, decision.alias_candidate_count,
    decision.spatial_candidate_count, decision.boundary_feature_count,
    decision.manual_candidate_codes, decision.exact_candidate_codes,
    decision.alias_candidate_codes, decision.spatial_candidate_codes,
    decision.manual_target_psgc_code, decision.manual_override_reason,
    decision.manual_source_reference, decision.manual_approved_by, decision.manual_approved_at,
    decision.project_source_snapshot_id, decision.project_source_ingest_run_id,
    decision.project_source_version, decision.project_source_modified_at,
    decision.project_run_id, decision.project_rule_version,
    decision.psgc_source_snapshot_id, decision.psgc_source_ingest_run_id,
    decision.psgc_version, decision.psgc_run_id, decision.psgc_rule_version,
    decision.psgc_source_load_ts,
    decision.boundary_source_snapshot_id, decision.boundary_source_ingest_run_id,
    decision.boundary_source_system, decision.boundary_psgc_version, decision.boundary_version,
    decision.boundary_version AS boundary_source_version,
    decision.boundary_source_load_ts,
    CASE WHEN COALESCE(decision.boundary_psgc_version, '') = COALESCE(decision.psgc_version, '')
         THEN 'VERSION_ALIGNED' ELSE 'VERSION_MISMATCH_VISIBLE' END AS boundary_version_status,
    decision.alias_version, decision.manual_mapping_version,
    decision.mapping_rule_version, decision.mapping_run_id AS run_id,
    decision.source_load_ts
FROM project_region_mapping_decisions AS decision
LEFT JOIN official_project_regions AS region ON decision.accepted_region_code = region.psgc_code
LEFT JOIN official_project_regions AS spatial_region ON decision.spatial_region_code = spatial_region.psgc_code;


## Summary

Created `02-silver.silver_project_region_map` at one canonical `silver_project` mapping result per deterministic mapping run. Every project remains present. Methods follow the approved precedence, `Central Office` remains non-geographic without a Gold key, unresolved and ambiguous outcomes retain candidate evidence, and name-versus-coordinate conflicts retain the higher-priority region plus the conflicting coordinate region. Project, PSGC, boundary, configuration, and rule lineage are retained. Requires Databricks execution.


In [0]:
%sql
SELECT
    mapping_method, match_status, match_quality,
    COUNT(*) AS project_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS project_percentage
FROM `02-silver`.silver_project_region_map
GROUP BY mapping_method, match_status, match_quality
ORDER BY project_rows DESC, mapping_method;
